# Model Evaluation — Visual Comparison

Generates comparison visuals from the artifacts already saved by the three
model notebooks (`output/*_predictions.csv`, `output/*_metrics.csv`) and the
model comparison notebook (`output/model_comparison.csv`). No retraining
happens here — this notebook only reads and plots.

Saves five PNGs to `../evaluation/`:
1. `1_roc_curve.png`
2. `2_pr_curve.png`
3. `3_metric_comparison.png`
4. `4_confusion_matrices.png`
5. `5_feature_importance_comparison.png`

## 0. Setup

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    auc,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
)

OUTPUT_DIR = Path("../output")
MODELS_DIR = Path("../models")
EVAL_DIR = Path("../evaluation")
EVAL_DIR.mkdir(exist_ok=True)

MODELS = {
    "Logistic Regression": "logistic_regression",
    "Random Forest": "random_forest",
    "XGBoost": "xgboost",
}
CLASS_ORDER = ["Healthy", "Stressed", "Critical"]
PROB_COLS = {"Healthy": "prob_healthy", "Stressed": "prob_stressed", "Critical": "prob_critical"}

In [ ]:
predictions = {
    name: pd.read_csv(OUTPUT_DIR / f"{slug}_predictions.csv")
    for name, slug in MODELS.items()
}
metrics_table = pd.read_csv(OUTPUT_DIR / "model_comparison.csv", index_col="Model")

print("Loaded predictions for:", list(predictions.keys()))
print("\nMetrics table:")
print(metrics_table.round(4).to_string())

## 1. ROC Curves (One-vs-Rest)

One panel per class. Each panel overlays all three models so you can compare
how well each model separates that class from the other two — the Critical
panel is the one that matters most for this project.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, cls in zip(axes, CLASS_ORDER):
    for name, df in predictions.items():
        y_true = (df["actual_condition"] == cls).astype(int)
        fpr, tpr, _ = roc_curve(y_true, df[PROB_COLS[cls]])
        ax.plot(fpr, tpr, label=f"{name} (AUC={auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_title(f"{cls} vs Rest")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(fontsize=8)

plt.suptitle("ROC Curves — One-vs-Rest by Class", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(EVAL_DIR / "1_roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", EVAL_DIR / "1_roc_curve.png")

## 2. Precision-Recall Curves

More informative than ROC under class imbalance — this is the chart most
likely to expose real differences on the minority Critical class that ROC-AUC
can hide.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, cls in zip(axes, CLASS_ORDER):
    for name, df in predictions.items():
        y_true = (df["actual_condition"] == cls).astype(int)
        precision, recall, _ = precision_recall_curve(y_true, df[PROB_COLS[cls]])
        ap = average_precision_score(y_true, df[PROB_COLS[cls]])
        ax.plot(recall, precision, label=f"{name} (AP={ap:.3f})")
    ax.set_title(f"{cls} — Precision-Recall")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

plt.suptitle("Precision-Recall Curves by Class", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(EVAL_DIR / "2_pr_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", EVAL_DIR / "2_pr_curve.png")

## 3. Metric Comparison

Grouped bar chart across all saved test-set metrics, one bar group per
metric, one color per model. Reads directly from `model_comparison.csv`
(notebook 04) rather than re-pivoting the per-model files.

In [ ]:
ax = metrics_table.T.plot(kind="bar", figsize=(12, 6))
ax.set_ylabel("Score")
ax.set_ylim(0.7, 1.0)
ax.set_title("Model Comparison — Test Set Metrics")
plt.xticks(rotation=30, ha="right")
plt.legend(title="Model")
plt.tight_layout()
plt.savefig(EVAL_DIR / "3_metric_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", EVAL_DIR / "3_metric_comparison.png")

## 4. Confusion Matrices

Side-by-side heatmaps on a shared color scale, so the three models are
visually comparable at a glance.

In [ ]:
fig, axes = plt.subplots(1, len(predictions), figsize=(5 * len(predictions), 4))

for ax, (name, df) in zip(axes, predictions.items()):
    cm = confusion_matrix(df["actual_condition"], df["predicted_condition"], labels=CLASS_ORDER)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
        vmin=0, vmax=max(cm.max() for cm in [
            confusion_matrix(d["actual_condition"], d["predicted_condition"], labels=CLASS_ORDER)
            for d in predictions.values()
        ]),
        ax=ax, cbar=False,
    )
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices (Test Set)", y=1.03, fontsize=13)
plt.tight_layout()
plt.savefig(EVAL_DIR / "4_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", EVAL_DIR / "4_confusion_matrices.png")

## 5. Feature Importance Comparison

Random Forest uses impurity-based importance; XGBoost uses gain-based
importance — these are different measures and not on a directly comparable
scale, so the two are shown as separate side-by-side panels rather than one
overlaid chart. Logistic Regression is excluded here since its coefficients
(signed, scaled) aren't the same kind of quantity as tree-based importances;
if you want it included, plot `abs(coefficient)` per class separately instead.

In [ ]:
rf_model = joblib.load(MODELS_DIR / "random_forest_model.pkl")
xgb_model = joblib.load(MODELS_DIR / "xgboost_model.pkl")

rf_importance = pd.Series(
    rf_model.named_steps["classifier"].feature_importances_,
    index=rf_model.named_steps["classifier"].feature_names_in_,
).sort_values()

xgb_importance = pd.Series(
    xgb_model.named_steps["classifier"].feature_importances_,
    index=xgb_model.named_steps["classifier"].feature_names_in_,
).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
rf_importance.plot(kind="barh", ax=axes[0], color="#4C72B0")
axes[0].set_title("Random Forest — Impurity-Based Importance")
axes[0].set_xlabel("Importance")

xgb_importance.plot(kind="barh", ax=axes[1], color="#DD8452")
axes[1].set_title("XGBoost — Gain-Based Importance")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.savefig(EVAL_DIR / "5_feature_importance_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", EVAL_DIR / "5_feature_importance_comparison.png")

## Summary

All five figures are now in `../evaluation/`:

- `1_roc_curve.png`
- `2_pr_curve.png`
- `3_metric_comparison.png`
- `4_confusion_matrices.png`
- `5_feature_importance_comparison.png`

These are the visuals for the README / report; the underlying numbers they're
built from already live in `output/model_comparison.csv` and the per-model
prediction/metrics CSVs.